# Ensemble — Stacked Qualifying Position Predictor

**Architecture**: Ridge meta-learner (positive weights) on out-of-fold (OOF) predictions from XGBoost + LightGBM + CatBoost.

**OOF generation**: Rolling-window (train on all prior years, predict each year from 2017–2025 one at a time) — avoids leakage into the meta-learner.

**Prerequisite**: Run `f1_train_xgboost.ipynb`, `f1_train_lightgbm.ipynb`, `f1_train_catboost.ipynb` first.

In [ ]:
import pandas as pd
import numpy as np
import joblib, json, os, warnings
warnings.filterwarnings('ignore')

PROC_CSV = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\notebooks\f1_processed.csv'
MDL_DIR  = r'C:\Users\CLL\OneDrive\Documents\GitHub\F1-Race-Predictor\models'
os.makedirs(MDL_DIR, exist_ok=True)

# ── Feature columns (leakage-free — no Q1/Q2/Q3 gaps, no appearance flags) ──
FEATURE_COLS = [
    "reg_disruption_index", "years_since_reg_change",
    "season_stage_ratio", "driver_career_races", "driver_q3_rate_season",
    "driver_circuit_q_pos_hist", "prior_year_q_pos_same_circuit",
    "team_rolling_q_pos_5r", "driver_rolling_race_pos_5r", "teammate_q_gap_season",
    "is_night_race", "is_street_circuit",
    "driver_cum_pts", "team_cum_pts",
    "driver_pts_gap_to_leader", "team_pts_gap_to_leader",
    "driver_q_vs_race_delta_5r",
    "circuit_altitude_m", "circuit_length_km", "num_corners", "num_drs_zones",
    "driver_age", "is_home_race",
    "fp1_gap", "fp2_gap", "fp3_gap",
    "is_wet_qualifying", "track_temp_avg", "air_temp_avg", "humidity_avg", "wind_speed_avg",
    "has_fp_data",          # explicit NaN flag for 2023+ only features
]
TARGET = "GridPosition"
# Rolling-window validation years (train on all prior, validate on this year)
VAL_YEARS  = [2021, 2022, 2023]
TEST_YEARS = [2024, 2025]

## Load Models & Data

In [ ]:
import lightgbm as lgb
from xgboost import XGBRegressor
from catboost import CatBoostRegressor, Pool
from sklearn.preprocessing import LabelEncoder

# Load base models
xgb_model = joblib.load(os.path.join(MDL_DIR, "xgb_quali.joblib"))
lgb_model  = joblib.load(os.path.join(MDL_DIR, "lgbm_quali.joblib"))
cb_model   = CatBoostRegressor()
cb_model.load_model(os.path.join(MDL_DIR, "catboost_quali.cbm"))

# Load data
df = pd.read_csv(PROC_CSV)
df["has_fp_data"] = df["fp1_gap"].notna().astype(int)
df = df[df[TARGET].notna()].sort_values(["Season","Round"]).reset_index(drop=True)

# XGB/LGB need label-encoded power_unit
le = LabelEncoder()
df["power_unit_enc"] = le.fit_transform(df["power_unit"].fillna("Unknown"))

XCOLS_XLG = FEATURE_COLS + ["power_unit_enc"]   # XGB + LGB
XCOLS_CB  = FEATURE_COLS + ["power_unit"]        # CatBoost

X_xlg = df[XCOLS_XLG]
X_cb  = df[XCOLS_CB].copy()
X_cb["power_unit"] = X_cb["power_unit"].fillna("Unknown")
y     = df[TARGET].values
seasons = df["Season"].values

print(f"Dataset: {len(df)} rows | {df['Season'].nunique()} seasons")

## Generate Out-of-Fold Predictions

For each year 2017–2025: train each model on all preceding years, predict that year. This gives a valid (leak-free) training signal for the meta-learner.

In [ ]:
# Out-of-fold predictions using rolling-window CV
# For each val_year: train on all prior years, predict val_year
OOF_YEARS = list(range(2017, 2026))   # 2015/2016 used only as training seed

oof_xgb = np.full(len(df), np.nan)
oof_lgb = np.full(len(df), np.nan)
oof_cb  = np.full(len(df), np.nan)

for val_yr in OOF_YEARS:
    tr_mask = (seasons >= 2015) & (seasons < val_yr)
    va_mask = seasons == val_yr
    if va_mask.sum() == 0:
        continue

    # XGBoost
    m_xgb = XGBRegressor(**joblib.load(os.path.join(MDL_DIR, "xgb_best_params.json"))
                         if False else xgb_model.get_params())
    # Use best params from saved JSON
    with open(os.path.join(MDL_DIR, "xgb_best_params.json")) as fp:
        xp = json.load(fp)
    xp.update({"tree_method":"hist","device":"cpu","random_state":42,
               "enable_categorical":True,"verbosity":0})
    m_xgb = XGBRegressor(**xp)
    m_xgb.fit(X_xlg[tr_mask], y[tr_mask])
    oof_xgb[va_mask] = m_xgb.predict(X_xlg[va_mask])

    # LightGBM
    with open(os.path.join(MDL_DIR, "lgbm_best_params.json")) as fp:
        lp = json.load(fp)
    lp.update({"random_state":42,"verbose":-1})
    m_lgb = lgb.LGBMRegressor(**lp)
    m_lgb.fit(X_xlg[tr_mask], y[tr_mask],
              categorical_feature=["power_unit_enc"],
              callbacks=[lgb.log_evaluation(0)])
    oof_lgb[va_mask] = m_lgb.predict(X_xlg[va_mask])

    # CatBoost
    with open(os.path.join(MDL_DIR, "catboost_best_params.json")) as fp:
        cp = json.load(fp)
    cp.update({"random_seed":42,"verbose":0,"loss_function":"RMSE"})
    m_cb = CatBoostRegressor(**cp)
    cat_idx = [XCOLS_CB.index("power_unit")]
    m_cb.fit(Pool(X_cb[tr_mask], y[tr_mask], cat_features=cat_idx))
    oof_cb[va_mask] = m_cb.predict(X_cb[va_mask])

    n = va_mask.sum()
    print(f"  {val_yr}: n={n} | "
          f"XGB={np.mean(np.abs(oof_xgb[va_mask]-y[va_mask])):.3f} | "
          f"LGB={np.mean(np.abs(oof_lgb[va_mask]-y[va_mask])):.3f} | "
          f"CB={np.mean(np.abs(oof_cb[va_mask]-y[va_mask])):.3f}")

# Assemble OOF matrix (drop NaN rows = 2015-2016 used only as train)
oof_mask = ~np.isnan(oof_xgb)
S_train = np.column_stack([oof_xgb[oof_mask], oof_lgb[oof_mask], oof_cb[oof_mask]])
y_oof   = y[oof_mask]

print(f"\nOOF matrix shape: {S_train.shape}")
for i, name in enumerate(["XGB","LGB","CB"]):
    mae = np.mean(np.abs(S_train[:,i] - y_oof))
    print(f"  {name} OOF MAE: {mae:.4f}")

## Train Meta-Learner

`Ridge(positive=True)` constrains weights to be non-negative so the ensemble can't short one model against another.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score

# Ridge meta-learner on OOF predictions
# Positive coefficients enforced via constraint (or use non-negative Ridge)
ridge = Ridge(alpha=1.0, positive=True)
cv_maes = -cross_val_score(ridge, S_train, y_oof,
                            cv=5, scoring="neg_mean_absolute_error")
print(f"Meta-learner CV MAE: {cv_maes.mean():.4f} ± {cv_maes.std():.4f}")

ridge.fit(S_train, y_oof)
print(f"\nMeta-learner weights:")
for name, coef in zip(["XGB","LGB","CB"], ridge.coef_):
    print(f"  {name}: {coef:.4f}")

## Final Evaluation on 2024–2025 Test Set

In [ ]:
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from scipy.stats import spearmanr

te_mask = np.isin(seasons, TEST_YEARS)
tr_mask_full = seasons <= 2023

# Retrain all 3 base models on full 2015-2023 training set
with open(os.path.join(MDL_DIR, "xgb_best_params.json")) as fp:
    xp = json.load(fp)
xp.update({"tree_method":"hist","device":"cpu","random_state":42,"enable_categorical":True,"verbosity":0})
xgb_f = XGBRegressor(**xp)
xgb_f.fit(X_xlg[tr_mask_full], y[tr_mask_full])

with open(os.path.join(MDL_DIR, "lgbm_best_params.json")) as fp:
    lp = json.load(fp)
lp.update({"random_state":42,"verbose":-1})
lgb_f = lgb.LGBMRegressor(**lp)
lgb_f.fit(X_xlg[tr_mask_full], y[tr_mask_full],
          categorical_feature=["power_unit_enc"],
          callbacks=[lgb.log_evaluation(0)])

with open(os.path.join(MDL_DIR, "catboost_best_params.json")) as fp:
    cp = json.load(fp)
cp.update({"random_seed":42,"verbose":0,"loss_function":"RMSE"})
cb_f = CatBoostRegressor(**cp)
cat_idx = [XCOLS_CB.index("power_unit")]
cb_f.fit(Pool(X_cb[tr_mask_full], y[tr_mask_full], cat_features=cat_idx))

# Generate test predictions
p_xgb = xgb_f.predict(X_xlg[te_mask])
p_lgb = lgb_f.predict(X_xlg[te_mask])
p_cb  = cb_f.predict(X_cb[te_mask])
S_test = np.column_stack([p_xgb, p_lgb, p_cb])
p_ens  = ridge.predict(S_test)
y_te   = y[te_mask]

print("=== Test Set Results (2024-2025) ===")
for name, preds in [("XGBoost", p_xgb), ("LightGBM", p_lgb), ("CatBoost", p_cb), ("Ensemble", p_ens)]:
    mae  = mean_absolute_error(y_te, preds)
    rmse = np.sqrt(np.mean((preds - y_te)**2))
    rho, _ = spearmanr(y_te, preds)
    print(f"  {name:10s}  MAE={mae:.4f}  RMSE={rmse:.4f}  Spearman={rho:.4f}")

# ── Plots ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Actual vs predicted scatter
axes[0].scatter(y_te, p_ens, alpha=0.3, s=12, color="#3498db")
lims = [1, 20]
axes[0].plot(lims, lims, "k--", lw=0.8)
axes[0].set_xlabel("Actual GridPosition")
axes[0].set_ylabel("Ensemble predicted")
axes[0].set_title("Ensemble: actual vs predicted (2024-2025)")

# MAE per position
pos_mae = {}
for pos in range(1, 21):
    mask = y_te == pos
    if mask.sum() > 0:
        pos_mae[pos] = mean_absolute_error(y_te[mask], p_ens[mask])
axes[1].bar(list(pos_mae.keys()), list(pos_mae.values()), color="#e74c3c")
axes[1].set_xlabel("Actual GridPosition")
axes[1].set_ylabel("MAE")
axes[1].set_title("MAE per grid position (harder to predict mid-field)")
plt.tight_layout(); plt.show()

## Save Ensemble

In [ ]:
# Save meta-learner and config
joblib.dump(ridge, os.path.join(MDL_DIR, "stacker_quali.joblib"))

# Save the 3 final base models under ensemble-specific names
xgb_f.save_model(os.path.join(MDL_DIR, "xgb_quali_final.json"))
joblib.dump(lgb_f, os.path.join(MDL_DIR, "lgbm_quali_final.joblib"))
cb_f.save_model(os.path.join(MDL_DIR, "catboost_quali_final.cbm"))

ensemble_meta = {
    "meta_learner": "Ridge(alpha=1.0, positive=True)",
    "base_models": ["XGBoost","LightGBM","CatBoost"],
    "feature_cols": XCOLS_XLG,
    "cat_col_cb": "power_unit",
    "target": TARGET,
    "train_seasons": "2015-2023",
    "test_seasons": "2024-2025",
}
with open(os.path.join(MDL_DIR, "ensemble_config.json"), "w") as f:
    json.dump(ensemble_meta, f, indent=2)

print("Saved stacker_quali.joblib, *_final models, ensemble_config.json")